# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`

This notebook provides a step-by-step guide to loading, exploring, and processing a FAIR² Croissant dataset from Kilifi County, Kenya, using the `mlcroissant` library.

---
### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant JSON-LD metadata URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print dataset information
print(f"Name: {metadata.name}")
print(f"Description: {metadata.description}\n")
print(f"Version: {metadata.version if hasattr(metadata, 'version') else ''}")
print(f"Published: {metadata.datePublished if hasattr(metadata, 'datePublished') else ''}\n")

## 2. Data Overview

This section reviews available record sets, fields, and their Croissant `@id`s.

We list all record sets, each with its `@id`, and enumerate their fields and data columns.

In [ ]:
# List all record sets and their fields
if hasattr(metadata, 'recordSets') or hasattr(metadata, 'recordSet'):
    # Support both 'recordSets' and 'recordSet' naming
    record_sets = getattr(metadata, 'recordSets', None) or getattr(metadata, 'recordSet', None)
else:
    record_sets = []

# In this dataset, mlcroissant parses as list of objects with ids
print("Available record sets and their field/column @ids:")
record_sets_list = []
for recset in dataset.record_sets:
    print(f"- RecordSet name: {recset.name}, @id: {recset.id}")
    record_sets_list.append(recset.id)
    print("  Fields:")
    for field in recset.fields:
        print(f"    - {field.name} (@id: {field.id}) [dataType: {field.data_type}]")
    print("  Columns:")
    for column in recset.columns:
        print(f"    - {column.name} (@id: {column.id}) [source: {column.source}]")
    print()
# Save the IDs for programmatic use below
record_sets_ids = record_sets_list

## 3. Data Extraction

Extract data from each record set into a pandas DataFrame for analysis. All entities are referenced using their `@id` field for reproducible and robust dataset handling.

In [ ]:
dataframes = {}

# For this dataset, let's extract all primary RecordSets
for record_set_id in record_sets_ids:
    # Load as DataFrame
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded record set: {record_set_id}  | Shape: {df.shape}")
    print(f"Columns: {df.columns.tolist()}\n")

# For demonstration, pick the first available record set for downstream EDA
main_record_set_id = record_sets_ids[0] if record_sets_ids else None

if main_record_set_id:
    print(f"First 5 records from '{main_record_set_id}':")
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)

Apply standard data filtering, normalization, and grouping. All field names referenced by their Croissant `@id` as per metadata listing above.

In [ ]:
import numpy as np

# Example: For the main record set, let's select a common mental health score field, e.g., GAD-7 total score.
# We'll auto-detect likely numeric fields by searching for 'score' in column name, or fallback to first numeric field.

df = dataframes[main_record_set_id]

# Identify likely candidates for EDA (use column @ids)
numeric_column = None
for col in df.columns:
    if 'score' in str(col).lower() or 'gad' in str(col).lower() or 'phq' in str(col).lower():
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_column = col
            break

if not numeric_column:
    # Fallback: pick the first numeric column
    num_cols = df.select_dtypes(include=[np.number]).columns
    if len(num_cols) > 0:
        numeric_column = num_cols[0]
    else:
        print("No numeric columns found for analysis.")
        numeric_column = None

if numeric_column is not None:
    print(f"Using numeric field for EDA: {numeric_column}")
    # Set threshold (median + 1 for demo purposes)
    threshold = df[numeric_column].median() + 1
    filtered_df = df[df[numeric_column] > threshold].copy()
    print(f"Filtered records with {numeric_column} > {threshold}:")
    display(filtered_df.head())

    # Normalize field
    filtered_df[f"{numeric_column}_normalized"] = (
        filtered_df[numeric_column] - filtered_df[numeric_column].mean()
    ) / filtered_df[numeric_column].std()
    print(f"Normalized {numeric_column} for filtered records:")
    display(filtered_df[[numeric_column, f"{numeric_column}_normalized"]].head())

    # Try grouping by a likely demographic field (e.g., gender, education, if present)
    group_candidates = [col for col in df.columns if any(x in col.lower() for x in ['gender', 'education', 'village'])]
    group_field = group_candidates[0] if group_candidates else None
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_column].mean().to_frame()
        print(f"Grouped mean {numeric_column} by {group_field}:")
        display(grouped_df.head())
else:
    print("No suitable numeric field found for EDA in this record set.")

## 5. Visualization

Visualize the distribution of the selected numeric field and its relationship to a demographic attribute, if present.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_column is not None:
    # Histogram of the chosen numeric column
    plt.figure(figsize=(6,4))
    sns.histplot(df[numeric_column].dropna(), bins=20, kde=True)
    plt.title(f'Distribution of {numeric_column}')
    plt.xlabel(numeric_column)
    plt.ylabel('Count')
    plt.show()

    # Boxplot by group field, if present
    if group_field:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=group_field, y=numeric_column, data=df.dropna(subset=[numeric_column, group_field]))
        plt.title(f'{numeric_column} by {group_field}')
        plt.xlabel(group_field)
        plt.ylabel(numeric_column)
        plt.xticks(rotation=30)
        plt.show()

## 6. Conclusion

- We explored the FAIR² Croissant mental health survey dataset using the `mlcroissant` library, referencing all structures by `@id`.
- We loaded all record sets, extracted their fields, and demonstrated filtering and normalization of mental health score data.
- Distributions and groupings by demographic fields provide first insights for downstream health or policy analyses.

Further steps could include more extensive feature engineering, anomaly checking, or export for downstream machine learning tasks.
